In [1]:
# plot_cna_ground_truth.ipynb

In [2]:
import numpy as np
import pandas as pd

In [3]:
cna_fn = '/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/data/cna_profile.tsv'
chrom_anno_fn = '/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/base/data/refseq/chrom_length.tsv'
out_fn = '/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/data/cna_profile.png'

# Load data

### CNA profile

In [4]:
def format_chrom(x):
    x = str(x)
    return x[3:] if x.startswith('chr') else x

df = pd.read_csv(cna_fn, sep = '\t', header = None)
df.columns = ['chrom', 'start', 'end', 'clone', 'cn0', 'cn1']

df['total_cn'] = df['cn0'] + df['cn1']
df['cna_type'] = 'neutral'
df.loc[df['total_cn'] > 2, 'cna_type'] = 'gain'
df.loc[df['total_cn'] < 2, 'cna_type'] = 'loss'
df.loc[(df['total_cn'] == 2) & (df['cn0'] != 1), 'cna_type'] = 'loh'

df_cna = df[['chrom', 'start', 'end', 'clone', 'cna_type']].copy()
df_cna['chrom'] = df['chrom'].map(format_chrom)
df_cna

,chrom,start,end,clone,cna_type
0,1,123400001,248956422,tumor,gain
1,4,50000001,190214555,tumor,loss
2,8,1,45200000,tumor,loss
3,8,45200001,145138636,tumor,gain
4,13,17700001,114364328,tumor,loh
5,17,1,25100000,tumor,loh


In [5]:
df_cna['clone'] = df_cna['clone'].map({
    'tumor': 'Tumor'
})
df_cna

,chrom,start,end,clone,cna_type
0,1,123400001,248956422,Tumor,gain
1,4,50000001,190214555,Tumor,loss
2,8,1,45200000,Tumor,loss
3,8,45200001,145138636,Tumor,gain
4,13,17700001,114364328,Tumor,loh
5,17,1,25100000,Tumor,loh


### Chromosome annotations

In [6]:
df_chrom = pd.read_csv(chrom_anno_fn, sep = '\t')
df_chrom.head()

,chrom,start,end,length
0,1,0,248956422,248956422
1,2,0,242193529,242193529
2,3,0,198295559,198295559
3,4,0,190214555,190214555
4,5,0,181538259,181538259


# Plot

In [7]:
import matplotlib
import matplotlib.pyplot as plt
import os
import pandas as pd
from matplotlib.patches import Rectangle, Patch


def format_chrom(x):
    x = str(x)
    return x[3:] if x.startswith('chr') else x


def plot_clonal_cna_states(
    df_cna,
    df_chrom,
    clone_order,
    out_fn = None,
    figsize = (6, 3),
    state_colors = None,
    clone_colors = None,
    fontsize = 7,
    plot_legend = False
):
    """Plot clonal CNA states (gain/loss/loh/neutral) from a TSV/CSV file.
    
    Parameters
    ----------
    df_cna : pandas.DataFrame
        CNA ground truth - A pandas.DataFrame with columns:
        chrom : str
            Chromosome.
        start : int
            Gene start position (1-based, inclusive).
        end : int
            Gene end position (1-based, inclusive).
        clone : str
            Clone label.
        cna_type : {'gain', 'loss', 'loh', 'neutral'}
            CNA type.
    df_chrom : pandas.DataFrame
        Chromosome annotation data - A pandas.DataFrame with columns:
        chrom : str
            Chromosome.
        length : int
            Length of the chromosome.
    out_fn : str or None
        If provided, saves the plot (e.g., 'cna_plot.pdf').
    figsize : tuple
        Figure size (width, height) in inches.
    state_colors : dict or None
        Keys are CNA type, values are hex colors for CNA states.
    """
    matplotlib.rcParams['font.family'] = 'sans-serif'
    
    # check args.
    df_cna = df_cna.drop_duplicates()
    df_cna['chrom'] = df_cna['chrom'].map(format_chrom)
    df_chrom['chrom'] = df_chrom['chrom'].map(format_chrom)
    assert np.all(df_cna['chrom'].isin(df_chrom['chrom']))
    
    if state_colors is None:
        state_colors = {   
            'gain': '#D14050',
            'loss': '#0078BB',
            'loh': '#40B5A8',
            'neutral': '#F0F0F0'
        }
    assert np.all(df_cna['cna_type'].isin(state_colors.keys()))
    
    if clone_colors is None:
        clones = df_cna['clone'].unique()
        clone_colors = {c:plt.cm.tab10(i % 10) for i, c in enumerate(clones)}
    assert np.all(df_cna['clone'].isin(clone_order))
    assert np.all([c in clone_colors for c in clone_order])


    # keep chrom 1-22 and sort by genomic positions.
    target_chroms = [str(i) for i in range(1, 23)]
    for chrom in target_chroms:
        assert chrom in df_chrom['chrom'].to_numpy()
    
    df_cna = df_cna.loc[df_cna['chrom'].isin(target_chroms)].copy()
    df_cna['chrom'] = pd.Categorical(
        df_cna['chrom'], categories = target_chroms, ordered = True)
    df_cna = df_cna.sort_values(['clone', 'chrom', 'start', 'end'])
    
    df_chrom = df_chrom.loc[df_chrom['chrom'].isin(target_chroms)].copy()
    df_chrom['chrom'] = pd.Categorical(
        df_chrom['chrom'], categories = target_chroms, ordered = True)
    df_chrom = df_chrom.sort_values(['chrom'])
    
    
    # cumulative start positions.
    chrom_order = df_chrom['chrom'].unique()
    cum_start = {}
    chrom_lengths = {}
    current = 0
    for chrom in chrom_order:
        cum_start[chrom] = current
        length = df_chrom.loc[df_chrom['chrom'] == chrom, 'length'].iloc[0]
        chrom_lengths[chrom] = length
        current += length
    genome_length = current


    # replace possible `inf` values in CNA profile and 
    # convert to 0-based coordinates.
    tmp = df_cna.merge(df_chrom[['chrom', 'end']], on = ['chrom'], how = 'left')
    tmp['end'] = tmp['end_x']
    idx = np.isinf(tmp['end'])
    tmp.loc[idx, 'end'] = tmp.loc[idx, 'end_y']
    df_cna = tmp[['chrom', 'start', 'end', 'clone', 'cna_type']].copy()
    
    df_cna['start'] = df_cna['start'] - 1


    # plot.
    #fig = plt.figure(figsize = figsize)
    #ax = fig.add_axes([0.05, 0.1, 0.9, 0.8])
    fig, ax = plt.subplots(figsize = figsize)

    clones = clone_order
    n_clones = len(clones)

    top_bar_height = 1.0
    clone_height = 5 if n_clones == 1 else 3.5
    top_bar_y = n_clones * clone_height


    # plot top chromosome bar.
    chrom_colors = ['#E0E0E0', '#CCCCCC']
    for i, chrom in enumerate(chrom_order):
        x = cum_start[chrom]
        w = chrom_lengths[chrom]
        color = chrom_colors[i % 2]
        rect = Rectangle(
            (x, top_bar_y), w, top_bar_height, 
            facecolor = color, edgecolor = 'none'
        )
        ax.add_patch(rect)
        font_size = fontsize
        if i >= 10:
            font_size = fontsize - 1
        if i <= 15 or i % 2 == 0:
            cx = x + w / 2
            ax.text(
                cx, top_bar_y + top_bar_height/2-0.08, chrom,
                ha = 'center', va = 'center', fontsize = font_size
            )

        
    # plot genome box.
    color = state_colors['neutral']
    for i, clone in enumerate(clones):
        y = (n_clones - i - 1) * clone_height
        for chrom in chrom_order:
            x = cum_start[chrom]
            w = chrom_lengths[chrom]
            rect = Rectangle(
                (x, y), w, clone_height, 
                facecolor = color, edgecolor = 'none'
            )
            ax.add_patch(rect)


    # plot clone rows.
    for i, clone in enumerate(clones[::-1]):
        y = (n_clones - i - 1) * clone_height
        df = df_cna[df_cna['clone'] == clone].copy()
        for j in range(df.shape[0]):
            chrom = df['chrom'].iloc[j]
            start = df['start'].iloc[j]
            end = df['end'].iloc[j]
            cna_type = df['cna_type'].iloc[j]
            color = state_colors[cna_type]
            x = start + cum_start[chrom]
            w = end - start
            rect = Rectangle(
                (x, y), w, clone_height, 
                facecolor = color, edgecolor = 'none'
            )
            ax.add_patch(rect)

    # formatting.
    ax.set_xlim(0, genome_length)
    ax.set_ylim(0, n_clones * clone_height + top_bar_height)
    ax.set_yticks([(n_clones - i - 0.5) * clone_height for i in range(n_clones)])
    ax.set_yticklabels([c.capitalize() for c in clones], fontsize = fontsize)
    #ax.set_yticks([])
    ax.set_ylabel(None)
    ax.set_xticks([])
    ax.set_xlabel(None)
    for spine in ax.spines.values():
        spine.set_visible(False)


    # legend and title.
    if plot_legend:
        font_size = fontsize - 1
        title_shift = 0.005
        ax.legend(
            handles = [
                Patch(facecolor = state_colors['gain'], label = 'Gain'),
                Patch(facecolor = state_colors['loss'], label = 'Loss'),
                Patch(facecolor = state_colors['loh'], label = 'LOH'),
                Patch(facecolor = state_colors['neutral'], label = 'Neutral')
            ],
            loc = 'upper left',
            bbox_to_anchor = (0.999, 0.99),
            frameon = False, 
            fontsize = font_size,
            title = 'CNA Type',
            title_fontsize = font_size,
            alignment = 'left'
        )
    plt.tight_layout()


    # save or show
    if out_fn:
        plt.savefig(out_fn, dpi = 300, bbox_inches = 'tight')
    else:
        plt.show()

    plt.close(fig)

In [8]:
plot_clonal_cna_states(
    df_cna,
    df_chrom,
    clone_order = ['Tumor'],
    out_fn = out_fn,
    figsize = (5.5, 1),
    state_colors = None,
    clone_colors = {'Tumor':plt.cm.tab10(1), 'Normal':plt.cm.tab10(0)},
    plot_legend = True
)